# Heart Disease Prediction Notebook

This notebook cleans the dataset, trains baseline and XGBoost models for binary heart disease prediction, and exports the tuned XGBoost pipeline used by the Flask app.


In [1]:
import pickle

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


In [2]:
df = pd.read_csv("dataset.csv")
df["thal"] = df["thal"].replace({"reversable defect": "reversible defect"})

feature_columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalch",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
]

X = df[feature_columns].copy()
y = (df["num"] > 0).astype(int)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(y.value_counts())
print("\nMissing values:")
print(X.isnull().sum())


Dataset shape: (920, 16)
Target distribution:
num
1    509
0    411
Name: count, dtype: int64

Missing values:
age           0
sex           0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
dtype: int64


In [3]:
df.head(5)

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversible defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
numeric_features = [column for column in feature_columns if column not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
rf_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=800,
                random_state=42,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))


Random Forest Accuracy: 0.8695652173913043
              precision    recall  f1-score   support

           0       0.88      0.82      0.85        82
           1       0.86      0.91      0.89       102

    accuracy                           0.87       184
   macro avg       0.87      0.86      0.87       184
weighted avg       0.87      0.87      0.87       184

[[67 15]
 [ 9 93]]


In [6]:
xgb_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                n_estimators=600,
                max_depth=4,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_weight=2,
                gamma=0.1,
                reg_lambda=2.0,
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
            ),
        ),
    ]
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))
print(confusion_matrix(y_test, xgb_pred))


XGBoost Accuracy: 0.8532608695652174
              precision    recall  f1-score   support

           0       0.88      0.78      0.83        82
           1       0.84      0.91      0.87       102

    accuracy                           0.85       184
   macro avg       0.86      0.85      0.85       184
weighted avg       0.86      0.85      0.85       184

[[64 18]
 [ 9 93]]


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X, y, cv=cv, scoring="accuracy", n_jobs=1)

print("5-fold CV Accuracy:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


5-fold CV Accuracy: [0.82065217 0.84782609 0.79891304 0.78804348 0.84782609]
Mean CV Accuracy: 0.8206521739130436


In [8]:
with open("xgboost_app_model.sav", "wb") as model_file:
    pickle.dump(xgb_model, model_file)

print("Saved model to xgboost_app_model.sav")


Saved model to xgboost_app_model.sav


## SQLite database

These cells create the same local database used by the Flask app. It stores registered users and saved prediction results in `heart_disease.db`.

In [9]:
import sqlite3
from pathlib import Path

DB_PATH = Path("heart_disease.db")

def get_db_connection():
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    return connection

def init_db():
    with get_db_connection() as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                password_hash TEXT NOT NULL,
                created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
            """
        )
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS predictions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER,
                patient_name TEXT,
                patient_email TEXT,
                age REAL NOT NULL,
                sex TEXT NOT NULL,
                cp TEXT NOT NULL,
                trestbps REAL NOT NULL,
                chol REAL NOT NULL,
                fbs TEXT NOT NULL,
                restecg TEXT NOT NULL,
                thalach REAL NOT NULL,
                exang TEXT NOT NULL,
                oldpeak REAL NOT NULL,
                slope TEXT NOT NULL,
                ca REAL NOT NULL,
                thal TEXT NOT NULL,
                prediction INTEGER NOT NULL,
                probability REAL NOT NULL,
                created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (user_id) REFERENCES users (id)
            )
            """
        )
        existing_columns = {
            row["name"]
            for row in connection.execute("PRAGMA table_info(predictions)").fetchall()
        }
        if "patient_name" not in existing_columns:
            connection.execute("ALTER TABLE predictions ADD COLUMN patient_name TEXT")
        if "patient_email" not in existing_columns:
            connection.execute("ALTER TABLE predictions ADD COLUMN patient_email TEXT")

init_db()
print(f"Database ready: {DB_PATH.resolve()}")


Database ready: C:\Users\danish\Desktop\heart disease\heart_disease.db


In [10]:
with get_db_connection() as connection:
    users_count = connection.execute("SELECT COUNT(*) FROM users").fetchone()[0]
    predictions_count = connection.execute("SELECT COUNT(*) FROM predictions").fetchone()[0]

print("Users saved:", users_count)
print("Predictions saved:", predictions_count)


Users saved: 3
Predictions saved: 8
